# 14. PMD DMR

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}clustering/merged/L2_hiconly/Epi-TPB/100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{indir}clustering/merged/L2_hiconly/Epi-TPB/mergehic_rocpr.npy'`  ·  _other_
- `f'{indir}clustering/merged/5kCG100k3C_summary.h5ad'`  ·  _joint summary obj_
- `f'{peak_ct}/dmr.split0.slop25kb.500b.{bw_list[0].split("/")[-1]}.CGN-Merge.tsv'`  ·  _DMR_
- `f'{peak_ct}/dmr.bed'`  ·  _DMR_
- `f'{peak_ct}/dmr.split1.slop250kb.5kb.{mc_ct.split("/")[-1]}.CGN-Merge.tsv'`  ·  _DMR_
- `f'{peak_ct}/dmr.split0.slop25kb.500b.{mc_ct.split("/")[-1]}.CGN-Merge.tsv'`  ·  _DMR_


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob
from concurrent.futures import ProcessPoolExecutor, as_completed

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import RegionDS

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
adata = anndata.read_h5ad(f'{indir}clustering/merged/L2_hiconly/Epi-TPB/100k3C_embed.h5ad')
adata

In [ ]:
label_reorder = np.load(f'{indir}clustering/merged/L2_hiconly/Epi-TPB/mergehic_rocpr.npy', allow_pickle=True)


In [ ]:
adata.obs['hic_L2'] = label_reorder.copy()


In [ ]:
tmp = adata.obs.copy()
tmp['group'] = 'SCT'
tmp.loc[tmp['hic_L2']=='c0', 'group'] = 'VCT'
tmp['group'] = tmp['group'] + '-' + tmp['Donor'].astype(str)
tmp['allcpath'] = '/gale/netapp/entex/ENTEx/allc_CGN/' + tmp.index + '.CGN-Both.allc.tsv.gz'
tmp[['allcpath', 'group']].to_csv('PMD_DMR/allclist_Epi-TPB_celltype.csv', index=False, header=False)


In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs


In [ ]:
leg = {'c1':{'Tc-Mem':[1,2,3,5,9,10,11,13,14], 'Th-Mem':[0,4,6,7,8,12]},
       'c15':{'Tc-Naive':[3,4], 'Th-Naive':[0,1,2,5,6,7,8,9,10,11,12]},
       'c7':{'B-Naive':[0,10,11], 'B-Mem':[1,2,3,4,5,6,7,8,9,12]}}


In [ ]:
meta_hema = meta.loc[meta['L1'].isin(leg.keys())]
meta_hema['celltype'] = ''
for l1 in leg:
    for ct in leg[l1]:
        selct = [f'c{i}' for i in leg[l1][ct]]
        selc = (meta_hema['L1']==l1) & (meta_hema['L2_any'].isin(selct))
        meta_hema.loc[selc, 'celltype'] = ct
        

In [ ]:
meta_hema_pbmc = meta_hema.loc[meta_hema['Tissue']=='PBMC']
meta_hema_pbmc['cell_group'] = meta_hema_pbmc['celltype'].astype(str) + '-' + meta_hema_pbmc['Donor'].astype(str)


In [ ]:
meta_hema_pbmc[['Donor', 'celltype']].value_counts().unstack()

In [ ]:
tmp = meta_hema_pbmc.loc[meta_hema_pbmc['celltype'].isin(['Tc-Mem', 'Th-Mem'])]
tmp['allcpath'] = f'{ENTEX_ROOT}/' + tmp.index + '.CGN-Both.allc.tsv.gz'
tmp[['allcpath', 'cell_group']].to_csv('PMD_DMR/allclist_Hema-Tmem_celltype.csv', index=False, header=False)


In [ ]:
for (ct,d), tmp in meta_hema_pbmc.groupby(['celltype', 'Donor']):
    if ('Naive' in ct) or ('Mem' in ct):
        tmp['allcpath'] = f'{ENTEX_ROOT}/allc/' + tmp.index + '.CGN-Both.allc.tsv.gz'
        np.savetxt(f'PMD_DMR/Hema-PBMC/allclist_{ct}_{d}.tsv', tmp['allcpath'], fmt='%s')


In [ ]:
# adata = anndata.read_h5ad(f'{indir}clustering/tissue/L1/PBMC/5kCG100k3C_embed.h5ad')
# adata

In [ ]:
# selc = [[6], [3,5,9,11], [0,4], [2,7,14], [1], [13]]
# leg = ['Tc-Naive', 'Th-Naive', 'Th-Mem', 'Tc-Mem', 'B-Naive', 'B-Mem']
# tmp = adata.obs.loc[adata.obs['leiden_cons'].str[1:].astype(int).isin(np.concatenate(selc))]
# tmp['leiden_merge'] = ''
# for xx,yy in zip(selc, leg):
#     selcell = tmp['leiden_cons'].str[1:].astype(int).isin(xx)
#     tmp.loc[selcell, 'leiden_merge'] = yy


In [ ]:
# tmp = tmp.loc[tmp['leiden_merge'].isin(['Th-Mem', 'Tc-Mem'])]
# tmp['group'] = tmp['leiden_merge'] + '-' + tmp['Donor'].astype(str)
# tmp['allcpath'] = '/gale/netapp/entex/ENTEx/allc_CGN/' + tmp.index + '.CGN-Both.allc.tsv.gz'
# tmp[['allcpath', 'group']].to_csv('PMD_DMR/allclist_Hema-Tmem_celltype.csv', index=False, header=False)


In [ ]:
import cooler
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
def expand_bed(input_file, window_size, window, split, min_split_size):

    dist = window_size * window
    dist_str = num2str(dist)
    ws_str = num2str(window_size)
    bed = pd.read_csv(input_file, sep='\t', header=None, index_col=None, usecols=[0,1,2], names=['chrom', 'start', 'end'])
    bed = bed.loc[bed['chrom'].isin(chrom_sizes.index)]

    if split==0:
        mid = ((bed['start'] + bed['end']) // 2).astype(int)
        bed['start'], bed['end'] = mid.copy(), mid.copy()
    bed['start'] = bed['start'] - dist
    bed['end'] = bed['end'] + dist
    bed = bed.loc[(bed['start']>0) & (bed['end']<bed['chrom'].map(chrom_sizes))]
    
    bed_new = []
    for idx,xx,yy,zz in bed.reset_index().values:
        if split>0:
            split_size = (zz-yy-2*dist) / split
            if split_size<min_split_size:
                continue
        for i in range(window):
            bed_new.append([xx, yy+window_size*i, yy+window_size*(i+1), f'{idx}_{i}'])
        # if (yy+dist)<(zz-dist):
        #     bed_new.append([xx, yy+dist, zz-dist])
        if split>0:
            for i in range(split):
                bed_new.append([xx, yy+dist+split_size*i, yy+dist+split_size*(i+1), f'{idx}_{window+i}'])
        for i in range(window):
            bed_new.append([xx, zz-dist+window_size*i, zz-dist+window_size*(i+1), f'{idx}_{window+split+i}'])

    print(len(bed_new))
    bed_new = pd.DataFrame(bed_new)
    bed_new[[1,2]] = np.around(bed_new[[1,2]], decimals=0).astype(int)
    bed_new.to_csv(input_file.replace('.bed',f'.split{split}.slop{dist_str}b.{ws_str}b.bed'), sep='\t', header=False, index=False)
    return dist_str, ws_str


In [ ]:
import os
import time

def num2str(num):
    if num>=1e6:
        num_str = f'{int(num//1e6)}m'
    elif num>=1e3:
        num_str = f'{int(num//1e3)}k'
    else:
        num_str = f'{num}'
    return num_str
        
def generate_flankmap(peak_group, mc_group, window_size=500, window=50, split=0, min_split_size=1):
    dist_str, ws_str = expand_bed(f'{peak_group}.bed', window_size=window_size, window=window, split=split, min_split_size=min_split_size)
    time.sleep(3)
    cmd = f'bigWigAverageOverBed {mc_group}.CGN-Merge.frac.bw {peak_group}.split{split}.slop{dist_str}b.{ws_str}b.bed {peak_group}.split{split}.slop{dist_str}b.{ws_str}b.{mc_group.split("/")[-1]}.CGN-Merge.tsv'
    os.system(cmd)
    return


In [ ]:
dmr_list = np.sort(glob(f'{indir}analysis/PMD_DMR/{group_name}/*_dmr'))
bw_list = glob(f'{indir}analysis/PMD_DMR/{group_name}/*.frac.bw')
bw_list = np.sort([xx.replace('.CGN-Merge.frac.bw', '') for xx in bw_list])
print(dmr_list, bw_list)

In [ ]:
cpu = 32
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for peak_ct in dmr_list:
        for mc_ct in bw_list:
            future = executor.submit(
                generate_flankmap,
                peak_group=f'{peak_ct}/dmr',
                mc_group=mc_ct,
                window_size=5000, window=50, split=1, min_split_size=1
            )
            futures[future] = f'{peak_ct}-{mc_ct}'
    result = {}
    for future in as_completed(futures):
        ct = futures[future]
        result[ct] = future.result()
        print(f'{ct} finished')
        

In [ ]:
idx, hypo, sample = [], [], []
for peak_ct in dmr_list:
    data = pd.read_csv(f'{peak_ct}/dmr.split0.slop25kb.500b.{bw_list[0].split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
    dmr_ds = RegionDS.open(peak_ct, region_dim='dmr')
    seldmr = data.index.str.split('_').str[0].astype(int).unique()
    dmr_bed = pd.read_csv(f'{peak_ct}/dmr.bed', sep='\t', index_col=None, header=None)
    seldmr = dmr_bed.loc[seldmr, 3].values
    dmr_data = dmr_ds.sel({'dmr':seldmr})['dmr_da_frac'].to_pandas().T
    idx.append(np.argsort(dmr_data.iloc[:, 0] - dmr_data.iloc[:, 1]))
    hypo.append(dmr_data.iloc[:, 0] < dmr_data.iloc[:, 1])
    sample.append(dmr_data.columns)
    

In [ ]:
idx = []
hypo = []
sample = []
for peak_ct,sel_ct in zip(dmr_list, [[0,2],[1,3],[0,1],[2,3]]):
    diff = []
    for mc_ct in bw_list[sel_ct]:
        data = pd.read_csv(f'{peak_ct}/dmr.split1.slop250kb.5kb.{mc_ct.split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
        ratio = data[5].values.reshape((-1, 101))
        cov = data[2].values.reshape((-1, 101))
        print((cov==0).sum() / cov.shape[0] / cov.shape[1])
        ratio[cov==0] = 1.0
        diff.append(ratio[:, 50])
    hypo.append(diff[0]<diff[1])
    idx.append(np.argsort(diff[0] - diff[1]))
    sample.append([xx.split('/')[-1] for xx in bw_list[[0,1]]])
    

In [ ]:
idx = []
hypo = []
sample = []
for peak_ct,sel_ct in zip(dmr_list, [[0,2],[1,3],[0,1],[2,3]]):
    diff = []
    for mc_ct in bw_list[sel_ct]:
        data = pd.read_csv(f'{peak_ct}/dmr.split0.slop25kb.500b.{mc_ct.split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
        ratio = data[5].values.reshape((-1, 100))
        cov = data[2].values.reshape((-1, 100))
        print((cov==0).sum() / cov.shape[0] / cov.shape[1])
        ratio[cov==0] = 1.0
        diff.append(ratio[:, 49:51].mean(axis=1))
    hypo.append(diff[0]<diff[1])
    idx.append(np.argsort(diff[0] - diff[1]))
    sample.append([xx.split('/')[-1] for xx in bw_list[[0,1]]])
    

In [ ]:
sample = []
for peak_ct,sel_ct in zip(dmr_list, [[0,2],[1,3],[0,1],[2,3]]):
    sample.append([xx.split('/')[-1] for xx in bw_list[sel_ct]])


In [ ]:
ave = []
fig, axes = plt.subplots(4, 4, figsize=(8,8), dpi=300, sharey='row')
for i,peak_ct in enumerate(dmr_list):
    avetmp = [[], []]
    for j,mc_ct in enumerate(bw_list):
        ax = axes[i, j]
        data = pd.read_csv(f'{peak_ct}/dmr.split0.slop25kb.500b.{mc_ct.split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
        ratio = data[5].values.reshape((-1, 100))
        cov = data[2].values.reshape((-1, 100))
        print((cov==0).sum() / cov.shape[0] / cov.shape[1])
        avetmp[0].append(np.nanmean(ratio[hypo[i]], axis=0))
        avetmp[1].append(np.nanmean(ratio[~hypo[i]], axis=0))
        ratio[cov==0] = 1.0
        if j==0:
            ax.set_ylabel(f'{ratio.shape[0]} {peak_ct.split("/")[-1].replace("_dmr"," DMRs")}')
        ax.imshow(ratio[idx[i]], cmap='jet', vmin=0, vmax=1.0, aspect='auto', 
                  interpolation='none', rasterized=True)
        ax.set_yticks([])
        ax.set_xticks([-0.5, 50.5, 100.5])
        ax.set_xticklabels(['-25k', 'Peak', '+25k'])
        if i==0:
            ax.set_title(f'{mc_ct.split("/")[-1]}')
    ave.append(avetmp)
    
plt.tight_layout()


In [ ]:
ave = []
fig, axes = plt.subplots(4, 4, figsize=(8,8), dpi=300, sharey='row')
for i,peak_ct in enumerate(dmr_list):
    avetmp = [[], []]
    for j,mc_ct in enumerate(bw_list):
        ax = axes[i, j]
        data = pd.read_csv(f'{peak_ct}/dmr.split0.slop25kb.500b.{mc_ct.split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
        ratio = data[5].values.reshape((-1, 100))
        cov = data[2].values.reshape((-1, 100))
        print((cov==0).sum() / cov.shape[0] / cov.shape[1])
        avetmp[0].append(np.nanmean(ratio[hypo[i]], axis=0))
        avetmp[1].append(np.nanmean(ratio[~hypo[i]], axis=0))
        ratio[cov==0] = 1.0
        if j==0:
            ax.set_ylabel(f'{ratio.shape[0]} {peak_ct.split("/")[-1].replace("_dmr"," DMRs")}')
        ax.imshow(ratio[idx[i]], cmap='jet', vmin=0, vmax=1.0, aspect='auto', 
                  interpolation='none', rasterized=True)
        ax.set_yticks([])
        ax.set_xticks([-0.5, 50.5, 100.5])
        ax.set_xticklabels(['-25k', 'Peak', '+25k'])
        if i==0:
            ax.set_title(f'{mc_ct.split("/")[-1]}')
    ave.append(avetmp)
    
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8,4), dpi=300, sharey='all', sharex='all')
for i,peak_ct in enumerate(dmr_list):
    for k in range(2):
        ax = axes[i//2, k+i%2*2]
        if k*2+i%2==0:
            ax.set_ylabel('mCG/CG')
        for j in range(4):
            ax.plot(np.arange(100), ave[i][k][j], color=f'C{j}', linewidth=1)
        ax.set_title(f'{sample[i][1-k]} hypo')

ax.set_xlim([-1, 100])
ax.set_xticks([0, 49.5, 99])
ax.set_xticklabels(['-25k', 'DMR', '+25k'])
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8,4), dpi=300, sharey='all', sharex='all')
for i,peak_ct in enumerate(dmr_list):
    for k in range(2):
        ax = axes[i//2, k+i%2*2]
        if k*2+i%2==0:
            ax.set_ylabel('mCG/CG')
        for j in range(4):
            ax.plot(np.arange(100), ave[i][k][j], color=f'C{j}', linewidth=1)
        ax.set_title(f'{sample[i][1-k]} hypo')

ax.set_xlim([-1, 100])
ax.set_xticks([0, 49.5, 99])
ax.set_xticklabels(['-25k', 'DMR', '+25k'])
plt.tight_layout()


In [ ]:
ave = []
fig, axes = plt.subplots(4, 4, figsize=(8,8), dpi=300, sharey='row')
for i,peak_ct in enumerate(dmr_list):
    avetmp = [[], []]
    for j,mc_ct in enumerate(bw_list):
        ax = axes[i, j]
        data = pd.read_csv(f'{peak_ct}/dmr.split1.slop250kb.5kb.{mc_ct.split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
        ratio = data[5].values.reshape((-1, 101))
        cov = data[2].values.reshape((-1, 101))
        print((cov==0).sum() / cov.shape[0] / cov.shape[1])
        avetmp[0].append(np.nanmean(ratio[hypo[i]], axis=0))
        avetmp[1].append(np.nanmean(ratio[~hypo[i]], axis=0))
        ratio[cov==0] = 1.0
        if j==0:
            ax.set_ylabel(f'{ratio.shape[0]} {peak_ct.split("/")[-1].replace("_dmr"," DMRs")}')
        ax.imshow(ratio[idx[i]], cmap='jet', vmin=0, vmax=1.0, aspect='auto', 
                  interpolation='none', rasterized=True)
        ax.set_yticks([])
        ax.set_xticks([-0.5, 50.5, 100.5])
        ax.set_xticklabels(['-250k', 'Peak', '+250k'])
        if i==0:
            ax.set_title(f'{mc_ct.split("/")[-1]}')
    ave.append(avetmp)
    
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8,4), dpi=300, sharey='all', sharex='all')
for i,peak_ct in enumerate(dmr_list):
    for k in range(2):
        ax = axes[i//2, k+i%2*2]
        if k*2+i%2==0:
            ax.set_ylabel('mCG/CG')
        for j in range(4):
            ax.plot(np.arange(101), ave[i][k][j], color=f'C{j}', linewidth=1)
        ax.set_title(f'{sample[i][k]} hypo')

ax.set_xlim([-1, 100])
ax.set_xticks([0, 50, 101])
ax.set_xticklabels(['-250k', 'DMR', '+250k'])
plt.tight_layout()


In [ ]:
ave = []
fig, axes = plt.subplots(4, 4, figsize=(8,8), dpi=300, sharey='row')
for i,peak_ct in enumerate(dmr_list):
    avetmp = [[], []]
    for j,mc_ct in enumerate(bw_list):
        ax = axes[i, j]
        data = pd.read_csv(f'{peak_ct}/dmr.split1.slop250kb.5kb.{mc_ct.split("/")[-1]}.CGN-Merge.tsv', sep='\t', header=None, index_col=0)
        ratio = data[5].values.reshape((-1, 101))
        cov = data[2].values.reshape((-1, 101))
        print((cov==0).sum() / cov.shape[0] / cov.shape[1])
        avetmp[0].append(np.nanmean(ratio[hypo[i]], axis=0))
        avetmp[1].append(np.nanmean(ratio[~hypo[i]], axis=0))
        ratio[cov==0] = 1.0
        if j==0:
            ax.set_ylabel(f'{ratio.shape[0]} {peak_ct.split("/")[-1].replace("_dmr"," DMRs")}')
        ax.imshow(ratio[idx[i]], cmap='jet', vmin=0, vmax=1.0, aspect='auto', 
                  interpolation='none', rasterized=True)
        ax.set_yticks([])
        ax.set_xticks([-0.5, 50.5, 100.5])
        ax.set_xticklabels(['-250k', 'Peak', '+250k'])
        if i==0:
            ax.set_title(f'{mc_ct.split("/")[-1]}')
    ave.append(avetmp)
    
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8,4), dpi=300, sharey='all', sharex='all')
for i,peak_ct in enumerate(dmr_list):
    for k in range(2):
        ax = axes[i//2, k+i%2*2]
        if k*2+i%2==0:
            ax.set_ylabel('mCG/CG')
        for j in range(4):
            ax.plot(np.arange(101), ave[i][k][j], color=f'C{j}', linewidth=1)
        ax.set_title(f'{sample[i][k]} hypo')

ax.set_xlim([-1, 100])
ax.set_xticks([0, 50, 101])
ax.set_xticklabels(['-250k', 'DMR', '+250k'])
plt.tight_layout()
